In [1]:
import open3d.ml as _ml3d
import open3d as o3d
import open3d.ml.torch as ml3d  # just switch to open3d.ml.tf for tf usage
import os

import warnings
warnings.filterwarnings("ignore")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
import torch
import open3d.ml.torch.ops as ops

print(ops.__file__)

/home/pointclouds/miniconda3/envs/open3dclone/lib/python3.10/site-packages/open3d/ml/torch/ops/__init__.py


In [3]:
import torch
print(torch.cuda.is_available())

True


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
cfg_file = "configs/pointtransformer_parislille3d.yml"
cfg = _ml3d.utils.Config.load_from_file(cfg_file)

In [6]:
device

device(type='cuda')

In [7]:
torch.cuda.get_device_capability()

(8, 6)

In [8]:
!python -c "import torch; print('PyTorch arch list:', torch.cuda.get_arch_list())"
!python -c "import open3d as o3d; print(dir(o3d.cuda))"

PyTorch arch list: ['sm_37', 'sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86']
['__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'pybind']


In [8]:
print(o3d.core.cuda.is_available())
import open3d.ml.torch.ops as ops
print('ops module:', dir(ops))
# Try executing a simple op on GPU to trigger the kernel
points = torch.randn(1000, 3).cuda()
print('Calling voxelize on GPU...')
try:
    result = ops.voxelize(
        points, 
        torch.zeros(1000, dtype=torch.int64).cuda(),
        torch.tensor([0.1, 0.1, 0.1]).cuda(),
        torch.tensor([-10.0, -10.0, -10.0]).cuda(),
        torch.tensor([10.0, 10.0, 10.0]).cuda()
    )
    print('voxelize OK')
except Exception as e:
    print('voxelize error:', e)# Try executing a simple op on GPU to trigger the kernel
points = torch.randn(1000, 3).cuda()
print('Calling voxelize on GPU...')
try:
    result = ops.voxelize(
        points, 
        torch.zeros(1000, dtype=torch.int64).cuda(),
        torch.tensor([0.1, 0.1, 0.1]).cuda(),
        torch.tensor([-10.0, -10.0, -10.0]).cuda(),
        torch.tensor([10.0, 10.0, 10.0]).cuda()
    )
    print('voxelize OK')
except Exception as e:
    print('voxelize error:', e)

True
ops module: ['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'ball_query', 'build_spatial_hash_table', 'continuous_conv', 'continuous_conv_transpose', 'fixed_radius_search', 'furthest_point_sampling', 'invert_neighbors_list', 'knn_search', 'nms', 'radius_search', 'ragged_to_dense', 'reduce_subarrays_sum', 'return_types', 'roi_pool', 'sparse_conv', 'sparse_conv_transpose', 'three_interpolate', 'three_interpolate_grad', 'three_nn', 'trilinear_devoxelize_backward', 'trilinear_devoxelize_forward', 'voxel_pooling', 'voxelize']
Calling voxelize on GPU...
voxelize OK
Calling voxelize on GPU...
voxelize OK


In [9]:
from open3d.ml.torch.datasets import ParisLille3D
import numpy as np

dataset = ParisLille3D(
    dataset_path="ParisLille3D",
    use_cache=False
)

split = dataset.get_split('training')
item = split.get_data(0)

print("Keys:", item.keys())

print("feat:", item['feat'])                         # None
print("feat type:", type(item.get('feat')))
print("feat value:", item.get('feat'))

print("point shape:", item.get('point').shape if item.get('point') is not None else None) #(N, 3)
print("feat shape:", item['feat'].shape) #(N, 3)
print("label shape:", item['label'].shape)           # (N,)
print("unique labels:", np.unique(item['label']))    # should NOT contain 12 in training batches

Keys: dict_keys(['point', 'feat', 'label'])
feat: [[-45.82838   99.59896    1.739715]
 [-50.783978  82.42716    2.586215]
 [-46.80768   77.34126    1.598015]
 ...
 [-35.130478 111.97026    0.869715]
 [-35.93798  110.04086   -2.137485]
 [-49.36068   96.07276   -2.093085]]
feat type: <class 'numpy.ndarray'>
feat value: [[-45.82838   99.59896    1.739715]
 [-50.783978  82.42716    2.586215]
 [-46.80768   77.34126    1.598015]
 ...
 [-35.130478 111.97026    0.869715]
 [-35.93798  110.04086   -2.137485]
 [-49.36068   96.07276   -2.093085]]
point shape: (31811099, 3)
feat shape: (31811099, 3)
label shape: (31811099,)
unique labels: [0 1 2 4 5 6 7 8]


In [7]:
dataset = ml3d.datasets.ParisLille3D(
    dataset_path='ParisLille3D',
    use_cache=False,
    val_files=[
        'Campus_BlockCb_group_6_inference_segmented.ply',
        'Campus_BlockCb_group_8_inference_segmented.ply'
    ]
)

print("Train files:", dataset.get_split_list('training'))
print("Val files:",   dataset.get_split_list('validation'))

Train files: ['ParisLille3D/training_10_classes/Campus_BlockCb_group_3_inference_segmented.ply', 'ParisLille3D/training_10_classes/Campus_BlockCb_group_5_inference_segmented.ply', 'ParisLille3D/training_10_classes/Campus_BlockCb_group_9_inference_segmented.ply']
Val files: ['ParisLille3D/training_10_classes/Campus_BlockCb_group_6_inference_segmented.ply', 'ParisLille3D/training_10_classes/Campus_BlockCb_group_8_inference_segmented.ply']


In [8]:
model = ml3d.models.PointTransformer(**cfg.model)

In [9]:
pipeline = ml3d.pipelines.SemanticSegmentation(model=model, dataset=dataset, device="cuda", **cfg.pipeline)

In [11]:
# import os

# ckpt_folder = "pretrained/"
# os.makedirs(ckpt_folder, exist_ok=True)
# ckpt_path = ckpt_folder + "kpconv_parislille3d_202011241550utc.pth"

In [12]:
'''
ADDED ENTRIES

num_workers: 0
pin_memory: False

TO THE yml FILE IN UNDER THE PIPELINE SECTION FOR THE RuntimeError OCCURED DUE TO COMPUTE CONSTRAINTS
'''

'\nADDED ENTRIES\n\nnum_workers: 0\npin_memory: False\n\nTO THE yml FILE IN UNDER THE PIPELINE SECTION FOR THE RuntimeError OCCURED DUE TO COMPUTE CONSTRAINTS\n'

## Training from scratch

In [10]:
%%time

pipeline.run_train()

validation: 100%|██████████| 1/1 [00:52<00:00, 52.09s/it]


CPU times: user 54.5 s, sys: 9.65 s, total: 1min 4s
Wall time: 13min 11s


## Retraining using checkpoints

In [ ]:
%%time

pipeline.load_ckpt(ckpt_path=ckpt_path, is_resume=True)
pipeline.run_train()

### Inference

In [15]:
import numpy as np
import torch

import glob

ckpt_files = sorted(glob.glob("./**/ckpt_*.pth", recursive=True))
print("Found checkpoints:")
for f in ckpt_files:
    print(" ", f)

ckpt_path = ckpt_files[-1]  # latest
print("\nLoading:", ckpt_path)
pipeline.load_ckpt(ckpt_path)
pipeline.model.to('cuda')
pipeline.model.eval()

# ── 2. Class names (0-8) ────────────────────────────────────────────────────
class_names = [
    'ground',
    'building',
    'pole',
    'bollard',
    'trash can',
    'barrier',
    'pedestrian',
    'car',
    'natural'
]
num_classes  = 9
IGNORE_LABEL = 12   # your noise label

# ── 3. Accumulate confusion matrix over all validation files ─────────────────
confusion_matrix = np.zeros((num_classes, num_classes), dtype=np.int64)
val_split        = dataset.get_split('validation')

print(f"Running inference on {len(val_split)} validation file(s)...\n")

with torch.no_grad():
    for idx in range(len(val_split)):
        data = val_split.get_data(idx)
        attr = val_split.get_attr(idx)
        print(f"  [{idx+1}/{len(val_split)}] {attr['name']}")

        true   = np.array(data['label']).flatten().astype(np.int32)

        result = pipeline.run_inference(data)        # ← only data, no attr
        pred   = np.array(result['predict_labels']).flatten().astype(np.int32)

        # mask out ignored label 12
        mask = true != IGNORE_LABEL
        pred = pred[mask]
        true = true[mask]

        valid = (
            (true >= 0) & (true < num_classes) &
            (pred >= 0) & (pred < num_classes)
        )
        np.add.at(confusion_matrix, (true[valid], pred[valid]), 1)

# ── 4. Compute per-class IoU and mIoU ───────────────────────────────────────
ious = []
for cls in range(num_classes):
    tp    = confusion_matrix[cls, cls]
    fp    = confusion_matrix[:, cls].sum() - tp
    fn    = confusion_matrix[cls, :].sum() - tp
    denom = tp + fp + fn
    ious.append(tp / denom if denom > 0 else float('nan'))

miou = np.nanmean(ious)

# ── 5. Print results ─────────────────────────────────────────────────────────
print("\n============================================")
print("        Validation Class-wise IoU           ")
print("============================================")
print(f"{'Class':<20} {'IoU':>10}  {'Points':>12}")
print("-" * 46)
for cls, (name, iou) in enumerate(zip(class_names, ious)):
    total  = confusion_matrix[cls, :].sum()
    iou_str = f"{iou*100:.2f}%" if not np.isnan(iou) else "  N/A"
    print(f"{name:<20} {iou_str:>10}  {total:>12,}")
print("-" * 46)
print(f"{'mIoU':<20} {miou*100:>9.2f}%")
print("============================================\n")

Found checkpoints:
  ./logs/PointTransformer_ParisLille3D_torch/checkpoint/ckpt_00000.pth
  ./logs/PointTransformer_ParisLille3D_torch/checkpoint/ckpt_00003.pth

Loading: ./logs/PointTransformer_ParisLille3D_torch/checkpoint/ckpt_00003.pth
Running inference on 2 validation file(s)...

  [1/2] Campus_BlockCb_group_6_inference_segmented


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument mat2 in method wrapper_mm)